# Alberta Electoral Boundary Audit — Interactive Explorer

**Phase 4C · Official Elections Alberta shapefiles · May 2026**

This notebook reproduces the key statistical findings from the audit without requiring a local checkout of the full repository (no large shapefiles, no MCMC re-run). It fetches the pre-computed output files directly from GitHub.

> **To run this notebook:** click **Runtime → Run all** (or press Ctrl+F9 on Windows / ⌘+F9 on Mac). All cells will execute in order. Results appear below each cell.

**Reproducibility note:** To pin to a specific commit, replace `master` in all URLs below with a commit SHA (e.g. `d8f101a`). The canonical outputs committed to `master` were generated by `packing_cracking_analysis.py` v0.3 and `mcmc_ensemble_canonical.py` with seed `1432864451` (1,010,000 plans across 4 chains).

---
Repository: https://github.com/Ixby/alberta-electoral-boundaries-audit  
Academic report: `reports/academic/report_academic.md`  
Pre-registration: OSF:6pt83, AsPredicted:#289,469, AsPredicted:#289,451

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import json, io, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BASE = 'https://raw.githubusercontent.com/Ixby/alberta-electoral-boundaries-audit/master'

def fetch_json(path):
    r = requests.get(f'{BASE}/{path}', timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_csv(path):
    r = requests.get(f'{BASE}/{path}', timeout=30)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

print('Libraries loaded.')

In [ ]:
# ── Load pre-computed outputs ─────────────────────────────────────────────────
phase4c   = fetch_json('data/outputs/phase4c_canonical_results.json')
real_map  = fetch_json('data/outputs/simulation_real_map_scores_canonical.json')
conv_diag = fetch_json('data/outputs/simulation_convergence_diagnostics_canonical.json')
ensemble  = fetch_csv('data/outputs/simulated_ensemble_percentiles_canonical.csv')

print('Files fetched.')
print(f'Ensemble rows: {len(ensemble):,}')
print(f'Ensemble columns: {list(ensemble.columns)}')

---
## 1. Key Findings Table

All numbers sourced directly from `phase4c_canonical_results.json` (S-M sign convention: positive EG = UCP structural advantage).

In [ ]:
maj = phase4c['majority']
mn  = phase4c['minority']

total_partisan = maj['total_ndp'] + maj['total_ucp']

maj_eg_pct    = maj['eg'] * 100
mn_eg_pct     = mn['eg']  * 100
maj_wasted    = round(maj['eg'] * total_partisan)
mn_wasted     = round(mn['eg']  * total_partisan)

maj_ndp_seats = round(maj['seats_at_50'] * maj['n_eds'])
mn_ndp_seats  = round(mn['seats_at_50']  * mn['n_eds'])

summary = pd.DataFrame([
    ['Structural vote-to-seat imbalance (efficiency gap)',
     f'+{maj_eg_pct:.2f}%', f'+{mn_eg_pct:.2f}%'],
    ['Excess wasted NDP votes (imbalance × total votes)',
     f'~{maj_wasted:,}', f'~{mn_wasted:,}'],
    ['NDP seats at a perfectly tied (50/50) provincial vote',
     f'{maj_ndp_seats} / 89', f'{mn_ndp_seats} / 89'],
    ['Mean-median difference',
     f"{maj['mean_median']*100:+.2f} pp", f"{mn['mean_median']*100:+.2f} pp"],
    ['Declination',
     f"{maj['declination']:+.4f}", f"{mn['declination']:+.4f}"],
], columns=['Metric', 'Majority map', 'Minority map'])
summary.set_index('Metric', inplace=True)
display(summary)

# ── Visual: headline comparison ───────────────────────────────────────────────
MAJORITY = '#1d6a27'
MINORITY  = '#922b21'

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
fig.suptitle('Key findings: majority vs. minority map', fontsize=11, fontweight='bold', y=1.02)

# Panel 1 – Efficiency gap
ax = axes[0]
ax.bar(['Majority', 'Minority'], [maj_eg_pct, mn_eg_pct],
       color=[MAJORITY, MINORITY], width=0.45, alpha=0.88)
ax.axhline(0, color='#333', lw=0.8)
ax.set_ylabel('Efficiency gap (%)')
ax.set_title('Structural vote imbalance\n(positive = UCP advantage)', fontsize=9)
for i, v in enumerate([maj_eg_pct, mn_eg_pct]):
    ax.text(i, v + 0.08, f'+{v:.2f}%', ha='center', fontsize=9, fontweight='bold')

# Panel 2 – Excess wasted votes
ax = axes[1]
ax.bar(['Majority', 'Minority'], [maj_wasted / 1000, mn_wasted / 1000],
       color=[MAJORITY, MINORITY], width=0.45, alpha=0.88)
ax.set_ylabel('Excess wasted NDP votes (thousands)')
ax.set_title('Extra NDP votes that\nchange no outcome', fontsize=9)
for i, v in enumerate([maj_wasted, mn_wasted]):
    ax.text(i, v / 1000 + 0.4, f'~{v:,}', ha='center', fontsize=9, fontweight='bold')

# Panel 3 – Seats at 50/50
ax = axes[2]
bars = ax.bar(['Majority', 'Minority'], [maj_ndp_seats, mn_ndp_seats],
              color=[MAJORITY, MINORITY], width=0.45, alpha=0.88)
ax.axhline(45, color='#555', lw=1.2, ls='--', label='45 = governing majority')
ax.set_ylim(38, 52)
ax.set_ylabel('NDP seats at a tied 50/50 vote')
ax.set_title('NDP seats if every voter\nsplit exactly 50/50', fontsize=9)
ax.legend(fontsize=8)
for i, v in enumerate([maj_ndp_seats, mn_ndp_seats]):
    ax.text(i, v + 0.3, str(v), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 2. Efficiency Gap — Maps vs. Neutral Ensemble

The histogram shows the distribution of efficiency-gap values across all plans in the 1,010,000-map neutral ensemble. Vertical lines mark where each commission map falls.

Positive values = UCP structural advantage. The audit's outlier threshold is the 95th percentile of the ensemble.

In [ ]:
# ── Efficiency gap: maps vs. ensemble ────────────────────────────────────────
eg = ensemble[ensemble['metric'] == 'efficiency_gap']
maj_eg_row = eg[eg['map'].str.contains('majority', case=False)].iloc[0]
mn_eg_row  = eg[eg['map'].str.contains('minority', case=False)].iloc[0]

maj_eg_pct = maj_eg_row['value']  * 100
mn_eg_pct  = mn_eg_row['value']   * 100
p5_eg      = maj_eg_row['ensemble_p5']  * 100
p50_eg     = maj_eg_row['ensemble_p50'] * 100
p95_eg     = maj_eg_row['ensemble_p95'] * 100
p_maj_eg   = maj_eg_row['percentile']
p_mn_eg    = mn_eg_row['percentile']

fig, ax = plt.subplots(figsize=(9, 2.2))

ax.axvspan(p5_eg, p95_eg, color='#aec6cf', alpha=0.35,
           label=f'Neutral ensemble p5–p95 ({p5_eg:.1f}% to {p95_eg:.1f}%)')
ax.axvline(p50_eg, color='#555', lw=1.2, ls='--',
           label=f'Neutral median ({p50_eg:.2f}%)')
ax.axvline(p95_eg, color='#888', lw=0.9, ls=':',
           label=f'p95 outlier line ({p95_eg:.2f}%)')

ax.scatter(maj_eg_pct, 0, s=160, color='#1d6a27', zorder=5,
           label=f'Majority map: +{maj_eg_pct:.2f}%  (p{p_maj_eg:.0f})')
ax.scatter(mn_eg_pct,  0, s=160, color='#922b21', zorder=5,
           label=f'Minority map: +{mn_eg_pct:.2f}%  (p{p_mn_eg:.0f})')

ax.set_xlabel('Efficiency gap  (positive = UCP structural advantage, %)')
ax.set_xlim(p5_eg - 0.5, max(mn_eg_pct, p95_eg) + 0.5)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_title('Efficiency gap: commission maps vs. 1,010,000-plan neutral ensemble')
ax.legend(fontsize=8.5, loc='upper left')
plt.tight_layout()
plt.show()

print(f'Majority map  p{p_maj_eg:.0f}  →  inside normal range (below the p95 outlier line)')
print(f'Minority map  p{p_mn_eg:.0f}  →  above the p95 outlier line')

---
## 3. Seats@50/50 — Maps vs. Neutral Ensemble

Holds the provincial vote at exactly 50/50 (NDP/UCP) and counts how many seats each map allocates to the UCP. A neutral Alberta map produces a median near 44.8% UCP seats (NDP has a slight efficiency advantage from urban concentration).

In [ ]:
# ── Seats@50/50: maps vs. ensemble ───────────────────────────────────────────
s50 = ensemble[ensemble['metric'] == 'seats_at_50_50']
maj_s50_row = s50[s50['map'].str.contains('majority', case=False)].iloc[0]
mn_s50_row  = s50[s50['map'].str.contains('minority', case=False)].iloc[0]

n_eds = maj['n_eds']

maj_ucp_frac = maj_s50_row['value']
mn_ucp_frac  = mn_s50_row['value']
p5_ucp       = maj_s50_row['ensemble_p5']
p50_ucp      = maj_s50_row['ensemble_p50']
p95_ucp      = maj_s50_row['ensemble_p95']
p_maj_s50    = maj_s50_row['percentile']
p_mn_s50     = mn_s50_row['percentile']

maj_ndp    = round((1 - maj_ucp_frac) * n_eds)
mn_ndp     = round((1 - mn_ucp_frac)  * n_eds)
neutral_ndp = round((1 - p50_ucp)     * n_eds)

to_pct = lambda f: f * 100

fig, ax = plt.subplots(figsize=(9, 2.2))

ax.axvspan(to_pct(p5_ucp), to_pct(p95_ucp), color='#c5d8e8', alpha=0.4,
           label=f'Neutral ensemble p5–p95  ({round((1-p5_ucp)*n_eds)}–{round((1-p95_ucp)*n_eds)} NDP seats)')
ax.axvline(to_pct(p50_ucp), color='#555', lw=1.2, ls='--',
           label=f'Neutral median: {neutral_ndp} NDP / {round(p50_ucp*n_eds)} UCP seats')
ax.axvline(to_pct(p95_ucp), color='#888', lw=0.9, ls=':',
           label='p95 outlier line')

ax.scatter(to_pct(maj_ucp_frac), 0, s=160, color='#1d6a27', zorder=5,
           label=f'Majority: {maj_ndp} NDP / {n_eds-maj_ndp} UCP  (p{p_maj_s50:.0f})')
ax.scatter(to_pct(mn_ucp_frac),  0, s=160, color='#922b21', zorder=5,
           label=f'Minority: {mn_ndp} NDP / {n_eds-mn_ndp} UCP  (p{p_mn_s50:.1f})')

ax.set_xlabel('UCP seat fraction at a perfectly tied 50/50 provincial vote (%)')
ax.set_xlim(to_pct(p5_ucp) - 1, to_pct(mn_ucp_frac) + 1)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_title('Seats at a 50/50 tied vote: commission maps vs. 1,010,000-plan neutral ensemble')
ax.legend(fontsize=8.5, loc='upper left')
plt.tight_layout()
plt.show()

majority_threshold = n_eds // 2 + 1
print(f'Majority needed to govern: {majority_threshold} seats')
print(f'Majority map:   {maj_ndp} NDP / {n_eds-maj_ndp} UCP  →  NDP majority  (p{p_maj_s50:.0f})')
print(f'Minority map:   {mn_ndp} NDP / {n_eds-mn_ndp} UCP  →  UCP majority  (p{p_mn_s50:.1f})')
print(f'Neutral median: ~{neutral_ndp} NDP / ~{round(p50_ucp*n_eds)} UCP')

---
## 4. Metric Summary — All Four Measures Side by Side

In [ ]:
# ── All metrics table ─────────────────────────────────────────────────────────
metrics_display = [
    ('efficiency_gap',       'Efficiency gap'),
    ('mean_median',          'Mean-median'),
    ('declination',          'Declination'),
    ('seats_at_50_50',       'Seats@50/50 (UCP frac)'),
    ('population_mad',       'Population MAD'),
]

rows = []
for metric_key, label in metrics_display:
    sub = ensemble[ensemble['metric'] == metric_key]
    maj_row = sub[sub['map'].str.contains('majority', case=False)]
    mn_row  = sub[sub['map'].str.contains('minority', case=False)]
    if maj_row.empty or mn_row.empty:
        rows.append({'Metric': label, 'Majority (raw)': 'n/a', 'Majority pctile': '—',
                     'Minority (raw)': 'n/a', 'Minority pctile': '—'})
        continue
    mr, nr = maj_row.iloc[0], mn_row.iloc[0]
    rows.append({
        'Metric':          label,
        'Majority (raw)':  f'{mr["value"]:.4f}' if pd.notna(mr['value']) else 'n/a',
        'Majority pctile': f'p{mr["percentile"]:.1f}' if pd.notna(mr['percentile']) else '—',
        'Minority (raw)':  f'{nr["value"]:.4f}' if pd.notna(nr['value']) else 'n/a',
        'Minority pctile': f'p{nr["percentile"]:.1f}' if pd.notna(nr['percentile']) else '—',
    })

df_metrics = pd.DataFrame(rows).set_index('Metric')
display(df_metrics)

# ── Visual: percentile dot chart across all metrics ───────────────────────────
# Each row = one metric. x-axis = ensemble percentile (0–100).
# Shaded band = p5–p95 (normal range). Dots = majority (green) / minority (red).
fig, axes = plt.subplots(len(metrics_display), 1,
                          figsize=(9, 1.5 * len(metrics_display)),
                          sharex=True)
fig.suptitle('All metrics: where each map sits in the 1,010,000-plan neutral ensemble',
             fontsize=10, fontweight='bold')

for ax, (metric_key, label) in zip(axes, metrics_display):
    sub = ensemble[ensemble['metric'] == metric_key]
    maj_row = sub[sub['map'].str.contains('majority', case=False)]
    mn_row  = sub[sub['map'].str.contains('minority', case=False)]

    ax.axvspan(5, 95, color='#cce0f0', alpha=0.45, label='Normal range (p5–p95)')
    ax.axvline(50, color='#666', lw=1, ls='--', label='Neutral median (p50)')
    ax.axvline(95, color='#aaa', lw=0.8, ls=':')

    if not maj_row.empty and pd.notna(maj_row.iloc[0]['percentile']):
        pct = maj_row.iloc[0]['percentile']
        ax.scatter(pct, 0, s=130, color='#1d6a27', zorder=5,
                   label=f'Majority  p{pct:.0f}')

    if not mn_row.empty and pd.notna(mn_row.iloc[0]['percentile']):
        pct = mn_row.iloc[0]['percentile']
        ax.scatter(pct, 0, s=130, color='#922b21', zorder=5,
                   label=f'Minority  p{pct:.1f}')

    ax.set_xlim(-2, 102)
    ax.set_ylim(-0.6, 0.6)
    ax.set_yticks([])
    ax.set_ylabel(label, fontsize=8.5, rotation=0, ha='right', labelpad=4)
    ax.legend(fontsize=7.5, loc='lower right', ncol=4,
              handletextpad=0.3, columnspacing=0.8)

axes[-1].set_xlabel('Ensemble percentile  (0 = most NDP-favourable; 100 = most UCP-favourable)')
plt.tight_layout()
plt.show()

---
## 5. MCMC Convergence Diagnostics

The ensemble is only valid if the Markov chains converged. Gelman-Rubin R̂ < 1.05 is the standard acceptance threshold; < 1.02 is excellent.

In [ ]:
# ── MCMC convergence diagnostics ─────────────────────────────────────────────
metric_labels = {
    'efficiency_gap':          'Efficiency gap',
    'mean_median':             'Mean-median',
    'declination':             'Declination',
    'seats_at_50_50':          'Seats@50/50',
    'population_mad':          'Population MAD',
    'reock_proxy_median':      'Reock (median)',
    'reock_proxy_pct_below_030': 'Reock (<0.30)',
}

metrics   = [k for k in conv_diag if isinstance(conv_diag[k], dict)]
n_effs    = [conv_diag[m]['n_eff'] for m in metrics]
taus      = [conv_diag[m]['tau']   for m in metrics]
rho1s     = [conv_diag[m]['rho_lag_1'] for m in metrics]
short_lbl = [metric_labels.get(m, m) for m in metrics]

# ── Visual ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
fig.suptitle('MCMC convergence diagnostics — 1,010,000-plan canonical ensemble',
             fontsize=10, fontweight='bold')

# Left: Effective Sample Size
ax = axes[0]
colors = ['#c0392b' if n < 500 else '#e67e22' if n < 1000 else '#1d6a27' for n in n_effs]
bars = ax.barh(short_lbl, n_effs, color=colors, alpha=0.85)
ax.axvline(1000, color='#555', lw=1.2, ls='--', label='n_eff = 1,000 (adequate)')
ax.axvline(500,  color='#999', lw=0.9, ls=':',  label='n_eff = 500 (minimal)')
ax.set_xlabel('Effective sample size (n_eff)')
ax.set_title('Effective sample size\n(higher = better-mixed chains)', fontsize=9)
ax.legend(fontsize=8)
for bar, val in zip(bars, n_effs):
    ax.text(val + 30, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', fontsize=8)

# Right: Lag-1 autocorrelation
ax = axes[1]
colors2 = ['#c0392b' if r > 0.995 else '#e67e22' if r > 0.99 else '#1d6a27' for r in rho1s]
bars2 = ax.barh(short_lbl, rho1s, color=colors2, alpha=0.85)
ax.axvline(0.99, color='#555', lw=1.2, ls='--', label='ρ₁ = 0.990 (high corr.)')
ax.set_xlim(0.97, 1.0)
ax.set_xlabel('Lag-1 autocorrelation (ρ₁)')
ax.set_title('Chain autocorrelation\n(lower = faster-mixing)', fontsize=9)
ax.legend(fontsize=8)
for bar, val in zip(bars2, rho1s):
    ax.text(val + 0.0001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

# ── Summary text ──────────────────────────────────────────────────────────────
print(f'Total plans: {conv_diag[metrics[0]]["n"]:,}')
print(f'\n{"Metric":<28} {"n_eff":>8}  {"τ (int. time)":>14}  {"ρ_lag1":>8}')
print('-' * 62)
for m, lbl in zip(metrics, short_lbl):
    d = conv_diag[m]
    print(f'{lbl:<28} {d["n_eff"]:>8,.0f}  {d["tau"]:>14,.1f}  {d["rho_lag_1"]:>8.4f}')

---
## 6. What This Does Not Show

This notebook reproduces the pre-computed outputs only. It does not:

- Re-run the MCMC ensemble (1,010,000 plans, ~6–8 hours on a 4-core laptop)
- Re-run the Phase 4C spatial vote attribution (requires the official Elections Alberta GeoPackage shapefiles)
- Reproduce the SZAT (swing-zone) or Mahalanobis joint tests

To fully reproduce from raw data, clone the repository and follow the step-by-step guide:  
[docs/REPRODUCING.md](https://github.com/Ixby/alberta-electoral-boundaries-audit/blob/master/docs/REPRODUCING.md)

---

## Disclosure

The author is a student at Mount Royal University. This research was conducted independently and was not commissioned as coursework. The author has in the past donated to and volunteered for the NDP; this is disclosed because it represents a potential source of motivated reasoning. The structural safeguard is symmetric methodology: every test was applied identically to both maps, and all tests were pre-registered before results were examined. This research received no external funding.

Pre-registration: **[OSF:6pt83](https://osf.io/6pt83)**, **AsPredicted:#289,469**, **AsPredicted:#289,451**  
Full academic report: [reports/academic/report_academic.md](https://github.com/Ixby/alberta-electoral-boundaries-audit/blob/master/reports/academic/report_academic.md)  
Repository: [github.com/Ixby/alberta-electoral-boundaries-audit](https://github.com/Ixby/alberta-electoral-boundaries-audit)